# Rotated Object Detection Export Example

This notebook provides an end-to-end example on how to train and export a **rotated** object detection model to a format accepted by uniVision Module Image ONNX. This will allow you to deploy your models on the Wenglor's B60 Smart Camera and MVC Machine Vision Controller.

Rotated object detection extends standard detection by predicting oriented bounding boxes (OBBs) — each box is described by a center point, width, height, and rotation angle. This is useful for objects that appear at arbitrary angles, such as aerial imagery, industrial parts on a conveyor, or text in documents.

DISCLAIMER: Focus of this notebook is to provide an example for the export and quantization. The training code is provided purely for illustration and not meant to provide a production-grade model. You will want to optimize the training pipeline and add more images. Depending on your situation, you can start from one of the following steps:
1. Train PyTorch model.
    * If you don't have a trained model, start here.
2. Export PyTorch model to ONNX.
    * If you already have a trained PyTorch model, start here.
3. Quantize ONNX model
    * If you have a trained ONNX model and it supports quantization, then start here. For this step, you will also need to provide a dataset that your model had been trained on. This dataset will be used for a calibration step during model quantization.
4. Export ONNX model.
    * If your ONNX model does not support quantization or you cannot provide a calibration dataset, begin here.

The models below have been tested and are known to run successfully:
* Rotated RTMDet-M-lite (default): mmrotate Rotated RTMDet-M with ReLU activations. The backbone, neck and most of the head are initialized from the RTMDet-M-lite COCO weights of the [TI edgeai model zoo](https://github.com/TexasInstruments/edgeai-tensorlab) (BSD-3-Clause / Apache-2.0); the class and angle layers are trained from scratch on your dataset.

We recommend a maximum input image size of 480×480 to stay within the memory and latency constraints of the B60 Smart Camera.

**What is quantization and why do we need it?**

Quantization is a technique used to make machine learning models run faster and more efficiently, especially on smaller devices with limited resources, like smartphones or smart cameras. By reducing the precision of the numbers the model uses, it can significantly decrease the amount of memory and computing power required, without losing too much accuracy. For example, our benchmark results show that a quantized ResNet50 model running on a B60 camera achieves an inference speed of 50 milliseconds, which is 8.4 times faster than the unquantized version (420 milliseconds).

**Annotation format**

This notebook expects annotations in COCO format where rotated bounding boxes are stored as 4-point segmentation polygons. This is the default export format from CVAT when annotating with rotated bounding boxes.

## 1. Train PyTorch model 

In [ ]:
# https://github.com/pytorch/pytorch/issues/140765
import getpass
import os

# Add dummy username so default_cache_dir() doesn't blow up
getpass.getuser = lambda: "torchuser"

# Also set the cache dir explicitly
os.environ["TORCHINDUCTOR_CACHE_DIR"] = "/tmp/torchinductor_cache"

In [ ]:
import copy
import json
import math
import os
import shutil
import time
import uuid
from functools import partial
from pathlib import Path

import cv2
import datumaro as dm
import mmcv
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
import requests
import torch
import torch.nn as nn
from mmcv.transforms import Compose
from mmdet.apis import inference_detector, init_detector
from mmengine.config import Config
from mmengine.runner import Runner
from mmrotate.evaluation.metrics.rotated_coco_metric import (
    RotatedCocoEval,
    qbox2rbox_list,
)
from mmrotate.utils import register_all_modules
from mmrotate.visualization import RotLocalVisualizer
from onnxruntime.quantization.quantize import (
    CalibrationMethod,
    QuantFormat,
    QuantType,
    quantize_static,
)
from onnxruntime.quantization.shape_inference import quant_pre_process
from pycocotools.coco import COCO
from sklearn.model_selection import train_test_split
from torchvision.ops import nms
from utils.bbox import visualize_rotated_bbox
from utils.enums import DatasetColorMode
from utils.export import export_univision_model_v4
from utils.image import (
    ensure_3ch_image,
    get_image_size,
    is_rgb_image,
    read_and_resize_input_example,
)
from utils.quantization import (
    TorchCalibrationDataReader,
    cap_channel_imbalance,
    find_final_convs,
    find_postprocess_nodes_to_exclude,
    keep_fp32_nodes_on_cpu,
    sort_nodes_topologically,
)
from utils.sparse_box_branch import (
    compute_box_branch_at_kept_candidates,
    quantize_features_with_relu_range,
)

register_all_modules()

In [ ]:
AI_INPUT_IMAGE_MAX_DIMENSION = 416
assert (
    AI_INPUT_IMAGE_MAX_DIMENSION % 32 == 0
), "AI_INPUT_IMAGE_SIZE must be divisible by 32"

# threshold to use to decide if a class probability enables the relative class
CLASS_THRESHOLD = 0.5

PAD_VALUE = (114, 114, 114)

DATA_ROOT = Path("../data")

ANNOTATION_FOLDER = DATA_ROOT / "coco-annotations/oriented"
EXAMPLE_IMAGE_FOLDER = DATA_ROOT / "images/multi-label/"
MODEL_FOLDER = DATA_ROOT / "model"

MMROTATE_DATA_FOLDER = DATA_ROOT / "mmrotate"
MMROTATE_OUTPUT_FOLDER = DATA_ROOT / "work_dirs" / "rotated_detection"

MMROTATE_DATA_FOLDER.mkdir(exist_ok=True)
MMROTATE_OUTPUT_FOLDER.mkdir(exist_ok=True, parents=True)
MODEL_FOLDER.mkdir(parents=True, exist_ok=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"The training will use {device} device")

In [ ]:
# this is to get around a change in pytorch default settings
torch.load = partial(torch.load, weights_only=False)

### 1.1 Prepare dataset

This notebook expects a COCO-format dataset exported from CVAT with rotated bounding box annotations.

**CVAT COCO 1.0 export format for rotated boxes:**
CVAT does NOT write rotated boxes as segmentation polygons. Instead it exports:
- `bbox`: `[x, y, w, h]` — the dimensions of the actual rotated box (not the AABB)
- `segmentation`: `[]` — empty
- `attributes.rotation`: rotation angle in degrees (clockwise)

Datumaro reads these as `dm.AnnotationType.bbox` objects with a `rotation` attribute. After the train/val split and datumaro re-export, we run a post-processing step to convert each `bbox + rotation` into a 4-point segmentation polygon. This polygon format is what mmrotate's data pipeline requires.

Set `ANNOTATION_FOLDER` to the path of your COCO annotation folder. The folder must contain an `annotations/` subfolder with a COCO JSON file and an `images/` subfolder with the corresponding images.

In [ ]:
coco_dataset = dm.Dataset.import_from(
    path=ANNOTATION_FOLDER,
    format="coco_instances",
)

In [ ]:
index_to_class = {v: k for k, v in coco_dataset.categories()[1]._indices.items()}
classes = list(index_to_class.values())
classes

In [ ]:
# For custom dataset, please skip this
# Copy images from our multilabel dataset to the coco dataset format
if ANNOTATION_FOLDER == DATA_ROOT / "coco-annotations/oriented":
    image_names = [Path(item.media.path).name for item in coco_dataset]
    image_source_paths = [
        next(EXAMPLE_IMAGE_FOLDER.rglob(image_name)) for image_name in image_names
    ]
    for image_source_path in image_source_paths:
        shutil.copy2(
            image_source_path, ANNOTATION_FOLDER / "images" / image_source_path.name
        )
else:
    print("Skipping image copy due to custom image dataset")

### 1.2 Visualize bounding boxes

Let's check the annotation to ensure that the dataset is read correctly.

In [ ]:
coco_dataset_list = list(coco_dataset)
datumaro_bbox_folder = MMROTATE_OUTPUT_FOLDER / "datumaro_bbox"
datumaro_bbox_folder.mkdir(exist_ok=True)

for item in coco_dataset_list:
    image_path = Path(item.media.path)

    # CVAT rotated bounding boxes are read by datumaro as dm.AnnotationType.bbox
    # with the rotation angle stored in annotation.attributes['rotation']
    bbox_annotations = [
        ann for ann in item.annotations if ann.type == dm.AnnotationType.bbox
    ]

    vis_image = visualize_rotated_bbox(
        image_path=image_path,
        boxes_xywhr=np.array(
            [
                [ann.x, ann.y, ann.w, ann.h, ann.attributes.get("rotation", 0.0)]
                for ann in bbox_annotations
            ]
        ),
        labels=np.array([ann.label for ann in bbox_annotations]),
        scores=np.array([1.0 for _ in bbox_annotations]),
        class_names=classes,
        score_threshold=CLASS_THRESHOLD,
        save_path=datumaro_bbox_folder / image_path.with_suffix(".jpg").name,
    )
print(f"Datumaro bounding boxes were visualized and saved to {datumaro_bbox_folder}")

### 1.3 Train-test-split

It is important to ensure that classes are equally distributed in train and validation. We will concatenate the class names so that we can do a stratified train-test-split. mmdetection expects the folder to be in the COCO format, we will transform our dataset to fit it.

```
dataset/
├── train/
│   ├── annotations/
│   │   └── instances_default.json
│   └── images/default/
│       ├── image1.jpg
│       ├── image2.jpg
│       └── ... (other image files)
└── valid/
    ├── annotations/
    │   └── instances_default.json
    └── images/default/
        ├── image1.jpg
        ├── image2.jpg
        └── ... (other image files)
```

In [ ]:
# Iterate over every image/item in the dataset
data = {}
for item in coco_dataset:
    image_id = item.id  # Usually the filename stem (without extension)
    # CVAT rotated bounding boxes are read by datumaro as dm.AnnotationType.bbox
    # with the rotation angle in annotation.attributes['rotation']
    label_counts = {}
    for ann in item.annotations:
        if ann.type == dm.AnnotationType.bbox:
            label_name = index_to_class[ann.label]
            label_counts[label_name] = label_counts.get(label_name, 0) + 1

    # Store in the main dict
    data[image_id] = label_counts

df = pd.DataFrame.from_dict(data, orient="index").fillna(0).astype(int)
column_names = list(df.columns)

missing_class = [class_i for class_i in classes if class_i not in column_names]
if missing_class:
    print(
        f"Warning! {missing_class} are missing but are included in the annotation categories!"
    )

df = df.reset_index(names="image_id")
df["label_combo"] = df[column_names].apply(
    lambda row: "".join(str(bool(x)) for x in row), axis=1
)

df.head(3)

In [ ]:
df[["label_combo"]].value_counts()

In [ ]:
try:
    train_image_id, valid_image_id = train_test_split(
        df.image_id.values,
        test_size=0.2,
        random_state=0,
        stratify=df.label_combo,
    )
except ValueError:
    # Stratified split requires at least 2 samples per class combination.
    # Falls back to a plain random split when the dataset is too small or all
    # images share the same label combination.
    print("Warning: stratified split not possible, falling back to random split")
    train_image_id, valid_image_id = train_test_split(
        df.image_id.values,
        test_size=0.2,
        random_state=0,
    )
train_image_id.shape, valid_image_id.shape

In [ ]:
train = copy.deepcopy(coco_dataset)
valid = copy.deepcopy(coco_dataset)

for image_id in valid_image_id:
    train.remove(image_id)

for image_id in train_image_id:
    valid.remove(image_id)

In [ ]:
train.export("train", format="coco_instances", save_media=True)
valid.export("valid", format="coco_instances", save_media=True)

In [ ]:
for phase in ["train", "valid"]:
    folder_path = MMROTATE_DATA_FOLDER / phase
    if folder_path.exists():
        print(f"{folder_path} exists. Copy is skipped")
    else:
        Path(phase).rename(folder_path)

In [ ]:
# Fix duplicate annotation IDs produced by Datumaro export
for phase in ["train", "valid"]:
    ann_path = MMROTATE_DATA_FOLDER / phase / "annotations" / "instances_default.json"
    with open(ann_path) as f:
        coco_json = json.load(f)
    for new_id, ann in enumerate(coco_json["annotations"], start=1):
        ann["id"] = new_id
    with open(ann_path, "w") as f:
        json.dump(coco_json, f)
    print(f"{phase}: re-numbered {len(coco_json['annotations'])} annotation IDs")

### 1.4 Normalize rotation encoding to COCO segmentation polygons

mmrotate's data pipeline reads rotated boxes from the `segmentation` field as 4-point polygons -- the convention most rotated-detection tooling (mmrotate, DOTA-style COCO datasets) uses to carry rotation through a COCO file, since plain COCO has no native rotated-box field.

CVAT's own COCO export doesn't use that convention: it leaves `segmentation` empty and stores the angle in a non-standard `attributes.rotation` field instead. This cell is a CVAT-specific adapter that normalizes CVAT's encoding into the segmentation-polygon form. A dataset that already provides segmentation polygons (e.g. converted from DOTA) passes through unchanged. A dataset using some other tool's encoding would need its own small adapter added here -- the cell raises an error rather than silently proceeding if it finds annotations with neither encoding.

In [ ]:
def cvat_bbox_rotation_to_polygon(x, y, w, h, angle_deg):
    """Convert a CVAT rotated bbox to a flat 4-point polygon.

    CVAT stores the rotated box as (x, y, w, h) — the unrotated box dimensions
    with top-left at (x, y) — plus a clockwise rotation angle in degrees applied
    around the box center (as seen on screen, y pointing down).

    Returns a flat list [x1, y1, x2, y2, x3, y3, x4, y4] suitable for the COCO
    segmentation field.
    """
    cx, cy = x + w / 2, y + h / 2
    hw, hh = w / 2, h / 2
    angle_rad = math.radians(angle_deg)
    cos_a = math.cos(angle_rad)
    sin_a = math.sin(angle_rad)

    corners = [(-hw, -hh), (hw, -hh), (hw, hh), (-hw, hh)]
    poly = []
    for dx, dy in corners:
        # Clockwise rotation in image coordinates (y pointing down):
        # [[cos, -sin], [sin, cos]] — same matrix as CCW in standard math coords
        rx = cx + cos_a * dx - sin_a * dy
        ry = cy + sin_a * dx + cos_a * dy
        poly.extend([rx, ry])
    return poly


def convert_rotation_annotations(ann_path):
    """CVAT-specific adapter: normalize CVAT's `attributes.rotation` encoding
    into the COCO segmentation-polygon convention this pipeline expects
    downstream. Annotations that already have a segmentation polygon (e.g. a
    dataset converted from DOTA) pass through unchanged.

    Returns (converted, unhandled). `unhandled` counts annotations with
    neither a CVAT rotation attribute nor an existing polygon -- i.e. a
    rotation encoding this notebook doesn't recognize yet.
    """
    with open(ann_path) as f:
        coco = json.load(f)

    converted = 0
    unhandled = 0
    for ann in coco["annotations"]:
        if ann.get("segmentation"):
            continue  # already in the expected polygon form
        rotation = ann.get("attributes", {}).get("rotation")
        if rotation is None:
            unhandled += 1
            continue
        x, y, w, h = ann["bbox"]
        poly = cvat_bbox_rotation_to_polygon(x, y, w, h, rotation)
        ann["segmentation"] = [poly]
        # Recompute bbox as AABB of the rotated polygon (required by COCO API)
        xs, ys = poly[0::2], poly[1::2]
        ann["bbox"] = [min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)]
        converted += 1

    with open(ann_path, "w") as f:
        json.dump(coco, f)
    return converted, unhandled


for phase in ["train", "valid"]:
    ann_path = MMROTATE_DATA_FOLDER / phase / "annotations" / "instances_default.json"
    converted, unhandled = convert_rotation_annotations(ann_path)
    print(
        f"{phase}: converted {converted} annotations to 4-point segmentation polygons"
    )
    if unhandled:
        raise ValueError(
            f"{phase}: {unhandled} annotation(s) have neither a CVAT "
            "`attributes.rotation` nor an existing segmentation polygon -- "
            "this dataset's rotation encoding isn't recognized. Add an "
            "adapter for it here, alongside cvat_bbox_rotation_to_polygon."
        )

### 1.5 Find image size with minimal padding

The accuracy of bounding box tends to improve when the image is not distorted, we will therefore resize the image while preserving the aspect ratio, the AI input image size will be achieved by padding. The following section will find the maximum image size of this dataset and calculate the image size with minimal padding based on the specified `AI_INPUT_IMAGE_MAX_DIMENSION` for optimal inference and training time. 

In [ ]:
image_sizes = []
for item in coco_dataset:
    image_sizes.append(get_image_size(item.media.path))
print(f"unique image sizes: {set(image_sizes)}")
image_sizes = np.array(image_sizes)

max_image_size = (np.max(image_sizes[:, 0]), np.max(image_sizes[:, 1]))

In [ ]:
def get_image_size_with_minimal_padding(
    original_size: tuple, max_dimension: int
) -> tuple:
    """
    Get image size with minimal padding that is divisible by 32 and has max_dimension as one dimension.

    Args:
        original_size: Original (width, height) tuple
        max_dimension: Maximum allowed dimension (e.g., 640, 512, 768)

    Returns:
        New (width, height) tuple that is divisible by 32 and minimizes padding
    """
    if max_dimension % 32 != 0:
        raise ValueError("Max dimension must be divisible by 32.")
    orig_height, orig_width = original_size
    aspect_ratio = orig_width / orig_height

    # Determine which dimension should be the max dimension
    if orig_height > orig_width:
        new_height = max_dimension
        new_width = int(np.ceil(new_height * aspect_ratio / 32) * 32)
    else:
        new_width = max_dimension
        new_height = int(np.ceil(new_width / aspect_ratio / 32) * 32)

    return (new_height, new_width)

In [ ]:
AI_INPUT_IMAGE_SIZE = get_image_size_with_minimal_padding(
    max_image_size, AI_INPUT_IMAGE_MAX_DIMENSION
)
print(f"AI_INPUT_IMAGE_SIZE: {AI_INPUT_IMAGE_SIZE}")

### 1.6 Train model

mmrotate trains models with a configuration-based system, for more details click [here](https://mmrotate.readthedocs.io/en/latest/). This notebook trains **Rotated RTMDet-M** with ReLU activations. It uses the stock mmrotate config with `act_cfg=ReLU`; the activation of its channel attention layers is replaced after the model is built (`use_relu_activations`).

In [ ]:
# Rotated RTMDet-M with ReLU activations, initialized from the RTMDet-M-lite COCO weights
# of the TI edgeai model zoo (BSD-3-Clause / Apache-2.0)
model_cfg = {
    "config": "rotated_rtmdet/rotated_rtmdet_m-3x-dota.py",
    "checkpoint": "rtmdet_m_coco_lite_640x640_20250404_checkpoint.pth",
    "url": "https://software-dl.ti.com/jacinto7/esd/modelzoo/11_02_00/models/vision/detection/coco/edgeai-mmdet/rtmdet_m_coco_lite_640x640_20250404_checkpoint.pth",
    # input preprocessing of the model
    "input_color_space": "BGR",
    "unit_scaling": False,
    "mean": (103.53, 116.28, 123.675),
    "std": (57.375, 57.12, 58.395),
}
default_model_config_path = Path(f"./mmrotate/configs/{model_cfg['config']}")
pretrained_checkpoint_path = MODEL_FOLDER / model_cfg["checkpoint"]


def use_relu_activations(model):
    """Uses ReLU in the channel attention layers of RTMDet (the config sets all other activations)."""
    for module in model.modules():
        if type(module).__name__ == "ChannelAttention":
            module.act = nn.ReLU()
    return model

In [ ]:
if not pretrained_checkpoint_path.exists():
    print("Downloading checkpoint")
    with requests.get(model_cfg["url"], stream=True) as response:
        response.raise_for_status()
        with open(pretrained_checkpoint_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
    print(f"Checkpoint downloaded successfully to {pretrained_checkpoint_path}")
else:
    print("Checkpoint already exists, skipping download")

In [ ]:
num_train_images = len(train_image_id)

BATCH_SIZE = min(8, num_train_images // 4)
steps_per_epoch = num_train_images // BATCH_SIZE

# Target ~5000 total training steps for finetuning
target_total_steps = 5000
EPOCHS = max(60, target_total_steps // steps_per_epoch)

# AdamW LR with linear scaling rule
base_lr = 0.00025
effective_lr = base_lr * (BATCH_SIZE / 16)
weight_decay = 0.01

backbone_lr_ratio = 0.1
neck_lr_ratio = 0.5
head_lr_ratio = 1.0

print(f"Dataset size:      {num_train_images} images")
print(f"Batch size:        {BATCH_SIZE}")
print(f"Steps per epoch:   {steps_per_epoch}")
print(f"Epochs:            {EPOCHS}")
print(f"Total steps:       {EPOCHS * steps_per_epoch}")
print(f"Effective LR:      {effective_lr}")
print(f"Weight decay:      {weight_decay}")
print(f"Backbone LR ratio: {backbone_lr_ratio}")
print(f"Neck LR ratio:     {neck_lr_ratio}")
print(f"Head LR ratio:     {head_lr_ratio}")

In [ ]:
img_scale = AI_INPUT_IMAGE_SIZE
annotation_json_relative_path = Path("annotations/instances_default.json")
images_relative_path = Path("images/default/")
data_root = str(MMROTATE_DATA_FOLDER)

In [ ]:
cfg = Config.fromfile(str(default_model_config_path))
cfg.backend_args = None

# --- Pipelines ---
train_pipeline = [
    dict(type="mmdet.LoadImageFromFile", backend_args=None),
    dict(type="mmdet.LoadAnnotations", with_bbox=True, with_mask=True, poly2mask=False),
    dict(type="ConvertMask2BoxType", box_type="rbox"),
    dict(type="mmdet.Resize", scale=img_scale, keep_ratio=True),
    dict(
        type="mmdet.RandomFlip",
        prob=0.75,
        direction=["horizontal", "vertical", "diagonal"],
    ),
    dict(type="mmdet.Pad", size=img_scale, pad_val=dict(img=PAD_VALUE)),
    dict(
        type="RandomRotate", prob=0.9, angle_range=180
    ),  # HSV jitter: hue/saturation/value shifts — image-only, safe with rotated boxes.
    dict(type="mmdet.YOLOXHSVRandomAug"),
    dict(type="mmdet.PackDetInputs"),
]

val_pipeline = [
    dict(type="mmdet.LoadImageFromFile", backend_args=None),
    dict(type="mmdet.Resize", scale=img_scale, keep_ratio=True),
    dict(type="mmdet.LoadAnnotations", with_bbox=True, with_mask=True, poly2mask=False),
    dict(type="ConvertMask2BoxType", box_type="qbox"),
    dict(type="mmdet.Pad", size=img_scale, pad_val=dict(img=PAD_VALUE)),
    dict(
        type="mmdet.PackDetInputs",
        meta_keys=(
            "img_id",
            "img_path",
            "ori_shape",
            "img_shape",
            "scale_factor",
            "instances",
        ),
    ),
]

inference_pipeline_cfg = [
    dict(type="mmdet.LoadImageFromFile", backend_args=None),
    dict(type="mmdet.Resize", scale=img_scale, keep_ratio=True),
    dict(type="mmdet.Pad", size=img_scale, pad_val=dict(img=PAD_VALUE)),
    dict(type="mmdet.PackDetInputs"),
]

# --- Dataloaders ---
cfg.train_dataloader = dict(
    batch_size=BATCH_SIZE,
    num_workers=4,
    persistent_workers=True,
    sampler=dict(type="DefaultSampler", shuffle=True),
    dataset=dict(
        type="mmdet.CocoDataset",
        data_root=data_root,
        ann_file=str(Path("train") / annotation_json_relative_path),
        data_prefix=dict(img=str(Path("train") / images_relative_path)),
        metainfo=dict(classes=classes),
        filter_cfg=dict(filter_empty_gt=True, min_size=32),
        pipeline=train_pipeline,
        backend_args=None,
    ),
)

cfg.val_dataloader = dict(
    batch_size=1,
    num_workers=4,
    persistent_workers=True,
    drop_last=False,
    sampler=dict(type="DefaultSampler", shuffle=False),
    dataset=dict(
        type="mmdet.CocoDataset",
        data_root=data_root,
        ann_file=str(Path("valid") / annotation_json_relative_path),
        data_prefix=dict(img=str(Path("valid") / images_relative_path)),
        metainfo=dict(classes=classes),
        test_mode=True,
        pipeline=val_pipeline,
        backend_args=None,
    ),
)
cfg.test_dataloader = cfg.val_dataloader

cfg.val_evaluator = dict(
    type="RotatedCocoMetric",
    metric="bbox",
    classwise=True,
    backend_args=None,
)
cfg.test_evaluator = cfg.val_evaluator

# --- Model overrides ---
cfg.model.bbox_head.num_classes = len(classes)

for part in ("backbone", "neck", "bbox_head"):
    cfg.model[part].act_cfg = dict(type="ReLU", inplace=True)
# the backbone is initialized from the checkpoint in `load_from`
cfg.model.backbone.init_cfg = None

# --- Training schedule ---
cfg.train_cfg.max_epochs = EPOCHS
cfg.train_cfg.val_interval = 1

# --- Optimizer: AdamW ---
cfg.optim_wrapper = dict(
    type="OptimWrapper",
    optimizer=dict(type="AdamW", lr=effective_lr, weight_decay=weight_decay),
    paramwise_cfg=dict(
        custom_keys={
            "backbone": dict(lr_mult=backbone_lr_ratio, decay_mult=1.0),
            "neck": dict(lr_mult=neck_lr_ratio, decay_mult=1.0),
            "bbox_head": dict(lr_mult=head_lr_ratio, decay_mult=1.0),
        },
        bypass_duplicate=True,  # RTMDet shares layers of its head across feature levels
    ),
    clip_grad=dict(max_norm=35, norm_type=2),
)

# --- LR scheduler: linear warmup + cosine annealing ---
cfg.param_scheduler = [
    dict(
        type="LinearLR",
        start_factor=1.0 / 3,
        by_epoch=False,
        begin=0,
        end=500,
    ),
    dict(
        type="CosineAnnealingLR",
        eta_min=effective_lr * 0.01,
        begin=0,
        T_max=EPOCHS,
        end=EPOCHS,
        by_epoch=True,
        convert_to_iter_based=True,
    ),
]

# --- Hooks ---
cfg.default_hooks.logger.interval = 50
cfg.default_hooks.checkpoint = dict(
    type="CheckpointHook",
    save_best="r_coco/bbox_mAP",
    rule="greater",
    interval=EPOCHS + 1,
    max_keep_ckpts=1,
    save_last=False,
)

cfg.custom_hooks = [
    dict(
        type="EMAHook",
        ema_type="mmdet.ExpMomentumEMA",
        momentum=0.0002,
        update_buffers=True,
        priority=49,
    ),
]

cfg.test_pipeline = inference_pipeline_cfg
cfg.work_dir = str(MMROTATE_OUTPUT_FOLDER)
cfg.load_from = str(pretrained_checkpoint_path)
cfg.resume = False

In [ ]:
runner = Runner.from_cfg(cfg)
use_relu_activations(runner.model)
runner.train()

### 1.7 Evaluate results

In [ ]:
config_file_path = MMROTATE_OUTPUT_FOLDER / default_model_config_path.name
checkpoint_file_path = sorted(
    list(MMROTATE_OUTPUT_FOLDER.glob("best_r_coco_bbox_mAP_epoch_*"))
)[-1]
model = init_detector(str(config_file_path), str(checkpoint_file_path), device="cpu")
use_relu_activations(model)
model.eval()
print("Model loaded")

In [ ]:
valid_images_dir = MMROTATE_DATA_FOLDER / "valid" / "images" / "default"

valid_image_paths = [
    p
    for p in valid_images_dir.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".avif"}
]

In [ ]:
raw_predictions_folder = MMROTATE_OUTPUT_FOLDER / "raw_prediction"
raw_predictions_folder.mkdir(exist_ok=True)

_inference_pipeline = Compose(inference_pipeline_cfg)

visualizer = RotLocalVisualizer()
visualizer.dataset_meta = model.dataset_meta

for image_path in valid_image_paths:
    result = inference_detector(model, image_path, test_pipeline=_inference_pipeline)

    img = mmcv.imread(image_path)
    visualizer.add_datasample(
        name="result",
        image=img,
        data_sample=result,
        draw_gt=False,
        show=False,
        pred_score_thr=CLASS_THRESHOLD,
    )

    vis_img = cv2.cvtColor(visualizer.get_image(), cv2.COLOR_RGB2BGR)
    cv2.imwrite(
        str(raw_predictions_folder / image_path.with_suffix(".jpg").name), vis_img
    )
print(f"Raw predictions were visualized and saved to {raw_predictions_folder}")

## 2. Export PyTorch Model to ONNX

If you already have a trained PyTorch model, start here.

**Note:** Quantizing for **NPU** acceleration may fail if your model contains unsupported layers/operators.  
Only the models listed at the start of this notebook are guaranteed to work in the full NPU workflow.

Alternatively, you can quantize and run on **CPU** instead. This still gives significant speed-ups with much broader layer support. See `inference_device` in [metadata.md](../metadata.md), this can be specifed at the final step of this notebook.

The module image ONNX expects the ONNX model output to be as follows:
* boxes: The shape of the corresponding output must be `(detections_count, 5)` for oriented boxes, the type is float32. The 5 values are `(cx, cy, w, h, angle_rad)` (see `boxes_format="center_size_angle"` in §4); the meaning is described by `boxes_format` and `boxes_coordinates`.
* labels: The shape of the corresponding output must be `(detections_count,)`, the type is int64. Each value must be in the range `[0..classes-1]`.
* scores: The shape of the corresponding output must be `(detections_count,)`, the type is float32. Each value must be in the range `[0..1]`.

In this example, we will create a wrapper that post-processes the output of the ONNX model to follow the above specifications.



In [ ]:
FP32_ONNX_MODEL_PATH = os.path.join(MODEL_FOLDER, "fp32_model.onnx")

In [ ]:
class RotatedDetectionExport(nn.Module):
    """Export wrapper for Rotated RTMDet (le90).

    Replicates ``bbox_head.predict_by_feat``: take the per-prior max-class sigmoid
    score, threshold, decode the boxes, then run NMS. The boxes are distances from
    each point prior to the box sides plus an angle, decoded with
    ``DistanceAnglePointCoder`` (only the kept priors).

    ``max_candidates`` limits the number of candidates above ``conf_thre`` to the
    best ones by score (like ``nms_pre`` in mmdet; candidates with the same score as
    the last one are kept as well). ``None`` keeps all candidates. The quantized
    model computes its last box layers per candidate (section 3), so the limit also
    bounds their cost.

    Output: boxes ``(N, 5)`` = ``(cx, cy, w, h, angle_rad)``, labels ``(N,)``
    int64, scores ``(N,)`` float32.
    """

    def __init__(
        self,
        base_model: nn.Module,
        conf_thre: float = 0.05,
        nms_iou_thr: float = 0.5,
        max_per_img: int = 100,
        max_candidates: int | None = 200,
    ):
        super().__init__()
        self.backbone = base_model.backbone
        self.neck = base_model.neck
        self.bbox_head = base_model.bbox_head
        self.conf_thre = conf_thre
        self.nms_iou_thr = nms_iou_thr
        self.max_per_img = max_per_img
        self.max_candidates = max_candidates
        self.num_classes = self.bbox_head.cls_out_channels

        if getattr(self.bbox_head, "share_conv", False):
            # RTMDet shares the convolutions of its head across feature levels, but has a
            # BatchNorm per level. Separate copies let the export fold each BatchNorm into
            # its convolution; the outputs are unchanged.
            for convs in (self.bbox_head.cls_convs, self.bbox_head.reg_convs):
                for level in range(1, len(convs)):
                    for layer in convs[level]:
                        layer.conv = copy.deepcopy(layer.conv)

        self.backbone.eval()
        self.neck.eval()
        self.bbox_head.eval()

    @staticmethod
    def _rbox_to_aabb(rboxes):
        """(N,5) [cx,cy,w,h,angle_rad] -> (N,4) [x1,y1,x2,y2] enclosing AABB.

        Inlined here so the export cell is self-contained and the op is
        traced into the ONNX graph.
        """
        cx, cy, w, h, a = rboxes.unbind(-1)
        cos, sin = a.cos().abs(), a.sin().abs()
        dx = w / 2 * cos + h / 2 * sin
        dy = w / 2 * sin + h / 2 * cos
        return torch.stack([cx - dx, cy - dy, cx + dx, cy + dy], dim=-1)

    @staticmethod
    def _flatten(tensors, last_dim):
        """Per-level head outputs (1, C, H, W) -> (sum(H*W*num_base), last_dim), in the
        (H, W, num_base) order of the priors (same as mmdet's internal predict_by_feat)."""
        return torch.cat(
            [t.permute(0, 2, 3, 1).reshape(-1, last_dim) for t in tensors]
        )

    def forward(self, x):
        feats = self.neck(self.backbone(x))
        outputs = self.bbox_head(feats)
        cls_scores, bbox_preds = outputs[0], outputs[1]

        featmap_sizes = [cs.shape[-2:] for cs in cls_scores]
        mlvl_priors = self.bbox_head.prior_generator.grid_priors(
            featmap_sizes,
            dtype=cls_scores[0].dtype,
            device=cls_scores[0].device,
        )

        scores, labels = self._flatten(cls_scores, self.num_classes).sigmoid().max(dim=1)
        keep = scores >= self.conf_thre  # confidence filter
        if self.max_candidates is not None and self.max_candidates < scores.numel():
            # only the best candidates; the mask keeps them in their original order
            kth_best_score = scores.topk(self.max_candidates).values[-1]
            keep = keep & (scores >= kth_best_score)
        scores, labels = scores[keep], labels[keep]

        # Point priors (x, y); distances to the 4 box sides and an angle per prior.
        priors = torch.cat(mlvl_priors)[keep]
        distances = self._flatten(bbox_preds, 4)[keep]
        angle_coder = self.bbox_head.angle_coder
        angles = angle_coder.decode(
            self._flatten(outputs[2], angle_coder.encode_size)[keep], keepdim=True
        )
        boxes = self.bbox_head.bbox_coder.decode(
            priors, torch.cat([distances, angles], dim=-1)
        )

        # Per-class AABB-NMS on AABB hulls (nms_rotated is not ONNX-exportable).
        aabbs = self._rbox_to_aabb(boxes)
        offset_step = aabbs.max() + 1
        class_offset = labels.to(aabbs).unsqueeze(1) * offset_step
        keep = nms(aabbs + class_offset, scores, self.nms_iou_thr)
        boxes, labels, scores = boxes[keep], labels[keep], scores[keep]

        if self.max_per_img > 0:
            boxes = boxes[: self.max_per_img]
            labels = labels[: self.max_per_img]
            scores = scores[: self.max_per_img]

        return boxes, labels.to(torch.int64), scores

In [ ]:
wrapper = RotatedDetectionExport(
    base_model=model,
    conf_thre=0.05,
    nms_iou_thr=0.5,
    max_per_img=100,
    max_candidates=200,
)

wrapper.eval()

# Export to ONNX
dummy_input = torch.randn(1, 3, AI_INPUT_IMAGE_SIZE[0], AI_INPUT_IMAGE_SIZE[1])
torch.onnx.export(
    wrapper,
    dummy_input,
    FP32_ONNX_MODEL_PATH,
    input_names=["input"],
    output_names=["boxes", "labels", "scores"],
    opset_version=20,
    do_constant_folding=True,
    dynamo=True,
    external_data=False,
    optimize=True,
)
print(f"ONNX FP32 model exported to {FP32_ONNX_MODEL_PATH}")

### 2.1 Evaluate the ONNX FP32 model

We will use the exported model to predict the validation dataset to verify the correctness of our wrapper.

In [ ]:
def preprocess_img_rotated(image, target_size):
    """Preprocess for the rotated detection models — matches training AND uniVision deploy.

    The export wrapper starts at ``backbone`` (it does NOT include mmdet's
    ``DetDataPreprocessor``), so the resize/pad/normalisation must be
    replicated here in the same ORDER the model was trained with:

    1. Resize keeping aspect ratio.
    2. Pad bottom/right to ``target_size`` with ``PAD_VALUE`` — **before**
       normalisation. Both the training pipeline (``mmdet.Pad`` runs before
       ``DetDataPreprocessor``) and uniVision (``resize_padding_value`` applied
       before unit scaling + standardisation, see §4.1.2) pad the *raw* image,
       so the pad region must be normalised too. Padding after normalisation
       with 0 would instead correspond to a raw pixel of ~ImageNet-mean, which
       the model never saw during training.
    3. BGR -> RGB if the model expects RGB (preprocessor with ``bgr_to_rgb=True``).
    4. Unit scaling (``/255``, if the model uses it) + standardisation with the
       model's mean and std (``model_cfg``).
    """
    h, w = image.shape[:2]

    # 1. Resize keeping aspect ratio.
    aspect_ratio = w / h
    if h > w:
        new_h = target_size[0]
        new_w = int(np.floor(aspect_ratio * new_h))
    else:
        new_w = target_size[1]
        new_h = int(np.floor(new_w / aspect_ratio))

    img = cv2.resize(image, (new_w, new_h))

    # 2. Pad bottom/right on the RAW image, before normalisation.
    pad_h = target_size[0] - new_h
    pad_w = target_size[1] - new_w
    img = cv2.copyMakeBorder(
        img, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=PAD_VALUE
    )

    # 3./4. Channel order, unit scaling and standardisation of the model.
    if model_cfg["input_color_space"] == "RGB":
        img = img[:, :, ::-1]
    img = img.astype(np.float32)
    if model_cfg["unit_scaling"]:
        img = img / 255
    mean = np.array(model_cfg["mean"], dtype=np.float32)
    std = np.array(model_cfg["std"], dtype=np.float32)
    img = (img - mean) / std

    return np.moveaxis(img, -1, 0).astype(np.float32)

In [ ]:
def create_onnx_session(model_path, providers=("CPUExecutionProvider",)):
    """Create an ONNX Runtime session with full graph optimisation enabled."""
    sess_options = ort.SessionOptions()
    sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(
        str(model_path), sess_options, providers=list(providers)
    )


def predict_rotated_onnx(session, image_bgr, target_size):
    """Run the exported rotated-detection ONNX model on a single BGR image.

    Returns ``(boxes, labels, scores, inference_time_s)``, where ``boxes`` is
    ``(N, 5)`` = ``(cx, cy, w, h, angle_rad)`` rescaled back to the ORIGINAL
    image coordinates. Only ``session.run`` is timed -- preprocessing and
    rescaling are excluded, so the number reflects model inference alone.
    """
    h, w = image_bgr.shape[:2]
    img_batch = np.expand_dims(preprocess_img_rotated(image_bgr, target_size), 0)

    input_name = session.get_inputs()[0].name
    output_names = [o.name for o in session.get_outputs()]

    start = time.monotonic()
    boxes, labels, scores = session.run(output_names, {input_name: img_batch})
    inference_time = time.monotonic() - start

    # Undo the resize: preprocess_img_rotated scales the dominant axis to
    # target_size (uniform aspect-ratio scale). Padding is bottom/right only, so
    # there is no offset to subtract. The angle is invariant to resize + pad, so
    # only cx, cy, w, h are rescaled.
    scale = target_size[0] / h if h > w else target_size[1] / w
    boxes = boxes.copy()
    boxes[:, :4] /= scale
    return boxes, labels, scores, inference_time


def rboxes_to_vis_xywhr(boxes):
    """(N, 5) ``(cx, cy, w, h, angle_rad)`` -> the ``[x_topleft, y_topleft, w, h,
    angle_deg]`` layout that ``visualize_rotated_bbox`` expects."""
    boxes_vis = boxes.copy()
    boxes_vis[:, 0] -= boxes_vis[:, 2] / 2
    boxes_vis[:, 1] -= boxes_vis[:, 3] / 2
    boxes_vis[:, 4] = np.degrees(boxes[:, 4])
    return boxes_vis


def save_onnx_predictions(model_path, image_paths, output_folder, label):
    """Predict over ``image_paths`` and save rotated-box visualisations."""
    output_folder.mkdir(exist_ok=True)
    session = create_onnx_session(model_path)

    total_inference_time = 0.0
    for image_path in image_paths:
        image = ensure_3ch_image(cv2.imread(str(image_path)))
        boxes, labels, scores, inference_time = predict_rotated_onnx(
            session, image, AI_INPUT_IMAGE_SIZE
        )
        total_inference_time += inference_time

        visualize_rotated_bbox(
            image_path=image_path,
            boxes_xywhr=rboxes_to_vis_xywhr(boxes),
            labels=labels,
            scores=scores,
            class_names=classes,
            score_threshold=CLASS_THRESHOLD,
            save_path=str(output_folder / image_path.with_suffix(".jpg").name),
        )

    print(f"Predictions of the {label} ONNX model were saved to {output_folder}.")
    print(f"Average inference time: {total_inference_time / len(image_paths)}")

In [ ]:
save_onnx_predictions(
    FP32_ONNX_MODEL_PATH,
    valid_image_paths,
    MMROTATE_OUTPUT_FOLDER / "fp32_onnx_predictions",
    label="FP32",
)

## 3. Quantize ONNX model (optional)
 
This step is required to utilize the NPU at B60, which dramatically speeds up model inference, the drawback is a potential degradation of model quality. This notebook uses a simple MinMax quantization with per-tensor weights, together with extra steps to preserve model quality.

Layers that are not quantized run on the CPU, the rest stays hardware-accelerated (`keep_fp32_nodes_on_cpu`).

**Note:** Percentile based quantization can preserve model quality better for some models, but requires significantly more time and memory to complete. 

We recommend excluding the NMS from quantization for 2 purposes:
* Preserve postprocessing quality
* Ensure NPU compability

In [ ]:
INT8_ONNX_MODEL_PATH = MODEL_FOLDER / "int8_model.onnx"
NUM_WORKERS = 4
CALIBRATION_SAMPLE_SIZE = 1024  # Please adjust this according to your usecase.

In [ ]:
class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, image_size):
        """
        Args:
            image_paths (list of str): List of file paths to JPEG images
            transform (callable, optional): Optional transform to be applied
        """
        # Filter out non-existent or invalid paths
        self.image_paths = [p for p in image_paths if os.path.exists(p)]
        self.image_size = image_size

        if len(self.image_paths) == 0:
            raise ValueError("No valid image paths found!")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]

        image = cv2.imread(image_path)  # mmdetection expects bgr images
        image = ensure_3ch_image(image)
        image = preprocess_img_rotated(image, AI_INPUT_IMAGE_SIZE)

        return image, 0

In [ ]:
train_images_dir = MMROTATE_DATA_FOLDER / "train" / "images" / "default"

train_image_paths = [
    p
    for p in train_images_dir.iterdir()
    if p.is_file()
    and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".avif"}
]

calibration_dataset = SimpleDataset(train_image_paths, AI_INPUT_IMAGE_SIZE)

In [ ]:
float32_capped_model_path = Path(
    str(FP32_ONNX_MODEL_PATH).replace(".onnx", "_capped.onnx")
)
float32_preprocessed_model_path = Path(
    str(FP32_ONNX_MODEL_PATH).replace(".onnx", "_preprocessed.onnx")
)

print("Quantizing ONNX model")

# Weights are quantized per tensor: cap the imbalance between the output channels of each
# layer, so that small channels keep enough precision. Where a capped layer is followed by
# ReLU and another convolution, the scaling is undone in that convolution.
onnx_model = onnx.load(FP32_ONNX_MODEL_PATH)
tensors_changed, channels_rescaled = cap_channel_imbalance(
    onnx_model, max_ratio=3.0, compensate=True
)
print(f"Rescaled {channels_rescaled} channels in {tensors_changed} layers")

# The last box layers are computed only at the candidates that pass the confidence
# threshold, so that they can stay in float32 at a small cost.
sparse_box_branch = compute_box_branch_at_kept_candidates(onnx_model)
onnx.save(onnx_model, float32_capped_model_path)

quant_pre_process(
    input_model_path=float32_capped_model_path,
    output_model_path=float32_preprocessed_model_path,
    skip_symbolic_shape=True,
)

# Auto-discover post-processing nodes (bbox decode + NMS) to exclude from
# quantization. This keeps the backbone+neck+head as one large quantized
# subgraph (ideal for NPU) while preserving fp32 precision for bbox
# coordinate math and NMS, which is critical for correct NMS behavior.

postprocess_nodes = find_postprocess_nodes_to_exclude(float32_preprocessed_model_path)
print(f"Excluding {len(postprocess_nodes)} post-processing nodes from quantization")

# The final class prediction layers stay in float32 as well (the box and angle prediction
# layers are part of the box branch computed above). The features for the box branch are
# computed by quantized layers.
prediction_nodes = [
    n
    for n in find_final_convs(float32_preprocessed_model_path)
    if n not in sparse_box_branch.feature_convs
]
print(f"Excluding {len(prediction_nodes)} prediction layers from quantization")

calibration_kwargs = {
    "dataset": calibration_dataset,
    "batch_size": 1,
    "shuffle": True,
    "num_workers": NUM_WORKERS,
    "persistent_workers": NUM_WORKERS > 0,
}
calibration_data_reader = TorchCalibrationDataReader(
    float32_preprocessed_model_path,
    samples=CALIBRATION_SAMPLE_SIZE,
    **calibration_kwargs,
)

quantize_static(
    model_input=float32_preprocessed_model_path,
    model_output=INT8_ONNX_MODEL_PATH,
    calibration_data_reader=calibration_data_reader,
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
    per_channel=False,
    calibrate_method=CalibrationMethod.MinMax,
    nodes_to_exclude=postprocess_nodes + prediction_nodes + sparse_box_branch.new_nodes,
)
# The layers excluded from quantization run on the CPU, the rest stays hardware-accelerated
onnx_model = onnx.load(INT8_ONNX_MODEL_PATH)
sort_nodes_topologically(onnx_model)
onnx_model = keep_fp32_nodes_on_cpu(onnx_model)
# the gathered features are quantized with the range of the ReLU that follows them
quantize_features_with_relu_range(onnx_model, sparse_box_branch.feature_convs)
onnx.save(onnx_model, INT8_ONNX_MODEL_PATH)
print(f"Model quantized and saved to: {INT8_ONNX_MODEL_PATH}")

### 3.1 Compare FP32 and INT8 models

Quantization might lead to a degradation in model quality, it is important to verify the quality of the quantized model by comparing it against the FP32 model. Here we would compare the COCO metrics pre and post quantization. Note that quantization is optional, if the model quality is not acceptable you can always run a FP32 model as long as it fits your inference time requirements.

In [ ]:
save_onnx_predictions(
    INT8_ONNX_MODEL_PATH,
    valid_image_paths,
    MMROTATE_OUTPUT_FOLDER / "int8_onnx_predictions",
    label="INT8",
)

In [ ]:
ann_file = str(
    MMROTATE_DATA_FOLDER / "valid" / "annotations" / "instances_default.json"
)
valid_images_dir = MMROTATE_DATA_FOLDER / "valid" / "images" / "default"

# RotatedCocoEval expects GT boxes as 5-value (cx, cy, w, h, angle_rad), but
# the CVAT export only has an axis-aligned `bbox` plus the rotated quad in
# `segmentation`. Convert that quad to the 5-value box before building the
# COCO index, matching RotatedCocoMetric.gt_to_coco_json.
with open(ann_file) as f:
    gt_dataset = json.load(f)
for ann in gt_dataset["annotations"]:
    rbox = qbox2rbox_list(ann["segmentation"][0])
    ann["bbox"] = rbox
    ann["area"] = rbox[2] * rbox[3]

coco_gt = COCO()
coco_gt.dataset = gt_dataset
coco_gt.createIndex()
img_id_to_info = {img["id"]: img for img in coco_gt.dataset["images"]}


def evaluate_onnx_model(model_path, label=""):
    """Run ONNX model on validation set and compute rotated COCO metrics."""
    session = create_onnx_session(
        model_path, providers=("CUDAExecutionProvider", "CPUExecutionProvider")
    )

    results = []

    for img_id, img_info in img_id_to_info.items():
        img_bgr = ensure_3ch_image(
            cv2.imread(str(valid_images_dir / img_info["file_name"]))
        )
        boxes, labels, scores, _ = predict_rotated_onnx(
            session, img_bgr, AI_INPUT_IMAGE_SIZE
        )

        for box, lbl, score in zip(boxes, labels, scores):
            results.append(
                {
                    "image_id": img_id,
                    "category_id": int(lbl) + 1,
                    "bbox": box.tolist(),  # [cx, cy, w, h, angle_rad]
                    "score": float(score),
                }
            )

    print(f"\n{'='*60}")
    print(f"  {label}: {model_path}")
    print(f"{'='*60}")
    if not results:
        print("  No detections!")
        return None

    coco_dt = coco_gt.loadRes(results)
    coco_eval = RotatedCocoEval(coco_gt, coco_dt, "bbox")
    coco_eval.params.maxDets = [100, 300, 1000]  # Match mmdetection's CocoMetric
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    return coco_eval.stats


fp32_stats = evaluate_onnx_model(FP32_ONNX_MODEL_PATH, "FP32")
int8_stats = evaluate_onnx_model(INT8_ONNX_MODEL_PATH, "INT8")

metric_names = [
    "AP @ IoU=0.50:0.95",
    "AP @ IoU=0.50",
    "AP @ IoU=0.75",
    "AP (small)",
    "AP (medium)",
    "AP (large)",
    "AR @ maxDets=100",
    "AR @ maxDets=300",
    "AR @ maxDets=1000",
    "AR (small)",
    "AR (medium)",
    "AR (large)",
]

print(f"\n{'='*60}")
print("  FP32 vs INT8 Comparison")
print(f"{'='*60}")
print(f"  {'Metric':<25} {'FP32':>8} {'INT8':>8} {'Δ':>8}")
print(f"  {'-'*25} {'-'*8} {'-'*8} {'-'*8}")
for name, fp32_val, int8_val in zip(metric_names, fp32_stats, int8_stats):
    delta = int8_val - fp32_val
    print(f"  {name:<25} {fp32_val:>8.4f} {int8_val:>8.4f} {delta:>+8.4f}")

## 4. Export ONNX model

### 4.1 YAML format

During model export, a metadata .yaml file is generated, it contains all the necessary information about the model such as the input size, class names, etc. It is essential to use the right parameters to ensure that correctness of the preprocessing. Please review the example metadata file and its description below. For more details, refer to [metadata.md](../metadata.md)

#### 4.1.1 `.yaml` metadata file example

#### 4.1.2 Input preprocessing order

1. (optional) Color space/channel transformation

    As a result of this transformation the image data has shape
    
        a. (batch_size, original_height, original_width, channels) for channel_order=NHWC
      
        b. (batch_size, channels, original_height, original_width) for channel_order=NCHW

   where channels is 1 or 3 and the image channel data matches the order defined by color_space (RGB/BGR/GRAYSCALE).

3. Image resize

4. (optional) Unit scaling.

    `x = x / 255`

   
5. (optional) Standardization

   `x = (x - standardization_mean) / standardization_std`

### 4.2 Export INT8 model

In [ ]:
UNIVISION_INT8_MODEL_PATH = MODEL_FOLDER / "example_int8_model.u3o"

In [ ]:
if any(is_rgb_image(path) for path in valid_image_paths):
    dataset_color_mode = DatasetColorMode.COLOR
else:
    dataset_color_mode = DatasetColorMode.MONOCHROME

dataset_color_mode

In [ ]:
input_example = read_and_resize_input_example(
    valid_image_paths[0], AI_INPUT_IMAGE_SIZE, dataset_color_mode
)

In [ ]:
export_univision_model_v4(
    univision_model_path=UNIVISION_INT8_MODEL_PATH,
    onnx_model_path=INT8_ONNX_MODEL_PATH,
    classes=classes,
    input_example=input_example,
    model_name=None,
    model_uuid=str(uuid.uuid4()),
    inference_device="AUTO",
    quantization="INT8",
    resize_mode="FIT_WITH_PADDING",
    resize_padding_value=PAD_VALUE,
    resize_image_alignment_horizontal="LEFT",
    resize_image_alignment_vertical="TOP",
    unit_scaling=model_cfg["unit_scaling"],
    standardization_std=model_cfg["std"],
    standardization_mean=model_cfg["mean"],
    channel_order="NCHW",
    dataset_color_mode=dataset_color_mode,
    input_color_space=model_cfg["input_color_space"],
    output_type="ROTATED_OBJECT_DETECTION",
    class_thresholds=[CLASS_THRESHOLD] * len(classes),
    boxes_output_index=0,
    labels_output_index=1,
    scores_output_index=2,
    boxes_format="center_size_angle",
    boxes_coordinates="absolute",
    angle_unit="radians",
    max_detections=20,
)

### 4.3 Export FP32 model

In [ ]:
UNIVISION_FP32_MODEL_PATH = MODEL_FOLDER / "example_fp32_model.u3o"

In [ ]:
if any(is_rgb_image(path) for path in valid_image_paths):
    dataset_color_mode = DatasetColorMode.COLOR
else:
    dataset_color_mode = DatasetColorMode.MONOCHROME

dataset_color_mode

In [ ]:
input_example = read_and_resize_input_example(
    valid_image_paths[0], AI_INPUT_IMAGE_SIZE, dataset_color_mode
)

In [ ]:
export_univision_model_v4(
    univision_model_path=UNIVISION_FP32_MODEL_PATH,
    onnx_model_path=FP32_ONNX_MODEL_PATH,
    classes=classes,
    input_example=input_example,
    model_name=None,
    model_uuid=str(uuid.uuid4()),
    inference_device="AUTO",
    quantization=None,
    resize_mode="FIT_WITH_PADDING",
    resize_padding_value=PAD_VALUE,
    resize_image_alignment_horizontal="LEFT",
    resize_image_alignment_vertical="TOP",
    unit_scaling=model_cfg["unit_scaling"],
    standardization_std=model_cfg["std"],
    standardization_mean=model_cfg["mean"],
    channel_order="NCHW",
    dataset_color_mode=dataset_color_mode,
    input_color_space=model_cfg["input_color_space"],
    output_type="ROTATED_OBJECT_DETECTION",
    class_thresholds=[CLASS_THRESHOLD] * len(classes),
    boxes_output_index=0,
    labels_output_index=1,
    scores_output_index=2,
    boxes_format="center_size_angle",
    boxes_coordinates="absolute",
    angle_unit="radians",
    max_detections=20,
)